# Ridgeline Precision: will the open orders ship on time?

This notebook is for the shop owner, the scheduler and the sales team. You do not need
to write code. Every gray box below is either a command you can run, or a file you can
read. Run the boxes from top to bottom. The answers are already saved under each box.

## (a) Your business

Ridgeline Precision is a 22-person CNC job shop in Wichita, Kansas. It makes parts to
order for three kinds of customers:

- **FTG-2201** hydraulic fittings: saw, then lathe, then inspect.
- **VB-3107** aluminum valve bodies: saw, then the Haas VF-2 mill (45 minutes each), then
  deburr, then inspect. They sell for $410 each.
- **MB-4410** mounting brackets: saw, then the older Fadal mill, then inspect.

The shop runs two shifts, 6 am to 10 pm, Monday to Friday. It is closed for Thanksgiving
and the day after. Two machinists run the saw, the lathe and the mills on each shift.
One inspector deburrs and inspects.

It is Monday morning, November 16. The ERP shows 15 open work orders due over the next
three weeks. Summit Valve, the biggest valve-body customer, charges $600 a day for each
late order. The owner has two ideas: hire a third machinist (about $68,000 a year), or buy
a second used VF-2 mill (about $95,000 installed). **The questions: which orders will be
late, what holds them up, and which idea fixes it?**

## (b) Your inputs

| File | What it is | Where it comes from |
|---|---|---|
| `data/open-orders.csv` | The open work orders, one per row | ERP open-order report |
| `shop.yaml` | The shop file: machines, people, shifts, routings, cycle times, prices | You write it once, from the routings and the time study |
| `ridgeline.twin.yaml` | The twin capsule: ties the shop file and the orders together | You write it once |
| `sweeps/machinist-or-mill.json` | The changes to test | See part (d) |

### The open orders

In [1]:
!cat data/open-orders.csv

work_order_id,customer,part,qty,start_date,due_date,priority
WO-26101,Cascade Hydraulics,FTG-2201,150,2026-11-16,2026-11-17,0
WO-26102,Summit Valve,VB-3107,40,2026-11-16,2026-11-18,0
WO-26103,Northline Trailers,MB-4410,60,2026-11-16,2026-11-18,0
WO-26104,Cascade Hydraulics,FTG-2201,120,2026-11-17,2026-11-19,0
WO-26105,Summit Valve,VB-3107,50,2026-11-17,2026-11-20,0
WO-26106,Prairie Ag Equipment,MB-4410,80,2026-11-18,2026-11-20,0
WO-26107,Delta Pumps,VB-3107,30,2026-11-18,2026-11-23,1
WO-26108,Northline Trailers,MB-4410,50,2026-11-19,2026-11-24,0
WO-26109,Cascade Hydraulics,FTG-2201,200,2026-11-19,2026-11-24,0
WO-26110,Summit Valve,VB-3107,45,2026-11-20,2026-11-25,0
WO-26111,Prairie Ag Equipment,MB-4410,70,2026-11-23,2026-11-30,0
WO-26112,Delta Pumps,VB-3107,35,2026-11-23,2026-12-01,0
WO-26113,Cascade Hydraulics,FTG-2201,140,2026-11-24,2026-12-02,0
WO-26114,Northline Trailers,MB-4410,40,2026-11-24,2026-12-03,0
WO-26115,Summit Valve,VB-3107,40,2026-11-30,2026-12-04,0


What each column means:

| Column | Meaning |
|---|---|
| `work_order_id` | The work order number from the ERP. |
| `customer` | Who ordered it. twinflow does not need it, but it helps you read the results. |
| `part` | The part number. It must match a part in the shop file. |
| `qty` | Pieces to make. |
| `start_date` | The day the material and the traveler are released to the floor. |
| `due_date` | The ship date promised to the customer. A date with no time means "by the end of that day". |
| `priority` | 1 for a rush order. A rush order jumps the queue at every machine. 0 is normal. |

Dates are counted from the shop's calendar start: Monday November 16, 6 am.

### The shop file

The shop file lists every machine, the people and their shifts, and the steps of each
part (the routing). A `rate` is pieces per second, and `load` is the setup time per job
in seconds. For example, `rate: 0.0003704` is one valve body every 45 minutes, and
`load: 2700` is a 45-minute setup for each job. `dispatch: edd` means the mill runs the
job that is due soonest first.

In [2]:
!cat shop.yaml

# Ridgeline Precision: the shop file.
# It lists the machines, the people and their shifts, the steps each part goes through
# (the routings), the cycle times and setup times, and what each part sells for.
# Times are in seconds. A rate is pieces per second: 0.0003704 is one piece every 45 minutes.
# `load` is the setup time for each job (fixtures, tools, first-piece check).

defaults:
  cycle_time_cv: 0.15        # real cycle times wander about 15% from the standard

stocks: []                   # raw bar, block and plate are always on the rack

part_types:
  - {name: steel_bar, uom: piece, attributes: {}}
  - {name: aluminum_block, uom: piece, attributes: {}}
  - {name: steel_plate, uom: piece, attributes: {}}
  - {name: FTG-2201_cut, uom: piece, attributes: {}}
  - {name: FTG-2201_turned, uom: piece, attributes: {}}
  - {name: FTG-2201, uom: piece, attributes: {}}        # hydraulic fitting
  - {name: VB-3107_cut, uom: piece, attributes: {}}
  - {name: VB-3107_milled, uom: piece, at

### The twin capsule

The capsule points at the shop file and the order book with `model: shop.yaml` and
`production_plan: data/open-orders.csv`. It also turns on state capture: the twin writes
down the state of the shop every 8 hours.

In [3]:
!cat ridgeline.twin.yaml

# Ridgeline Precision: the twin capsule. It ties the shop file and the order book together,
# and says how to run them. Open it with: twinflow scenario run ridgeline.twin.yaml
schema_version: '0.1'
required_capabilities:
  - manufacturing.basic
model: shop.yaml                          # machines, people, shifts, routings, money
snapshot:
  id: ridgeline-2026-11-16
  as_of: '2026-11-16T06:00:00-06:00'      # the order book as of Monday 6 am
  model_revision: '1'
  production_plan: data/open-orders.csv   # the open work orders, exported from the ERP
experiment:
  id: ridgeline-two-weeks
  replications: 5
  seed: 11
  state_capture:                          # write down the state of the shop every 8 hours
    every_hours: 8
assumptions:
  - Cycle times are the shop's standards with a 15% spread.
  - Raw stock is always on the rack.
  - The valve-body mill and the bracket mill are separate machines.
provenance:
  source: Ridgeline Precision ERP export, Monday 2026-11-16 06:00


## (c) Check and run

First, check the files. No message means the file is good.

In [4]:
!twinflow validate shop.yaml

In [5]:
!twinflow scenario validate ridgeline.twin.yaml

### Completion dates and on-time risk

This runs the two weeks 20 times. Each run draws cycle times that wander a little from
the standard, the way a real shop does. `--orders` prints one line per order.

- **Done (typical)** is the middle finish time over the 20 runs.
- **Done (bad case)** is the time by which 9 runs in 10 are done.
- **Late in** counts the runs in which the order shipped late.

The first lines describe run 1 only. The table and the busiest work centers use all 20.

In [6]:
!twinflow run shop.yaml --plan data/open-orders.csv --reps 20 --orders

run <run-id-1>
  orders:   15/15 complete  |  on-time 12 (80%)  |  late 3 (rep 1 of 20)
  makespan: 447.1 h
  dir:    ./runs/<run-id-1>
  report: ./runs/<run-id-1>/report.html
  kpis:   ./runs/<run-id-1>/kpis.json
  view:   twinflow report <run-id-1> --out html

Orders over 20 runs:
Order     Part      Qty  Due               Done (typical)    Done (bad case)   Late in
WO-26101  FTG-2201  150  Tue Nov 17 24:00  Mon Nov 16 21:51  Tue Nov 17 07:42  0 of 20
WO-26102  VB-3107   40   Wed Nov 18 24:00  Wed Nov 18 14:49  Wed Nov 18 17:58  0 of 20
WO-26103  MB-4410   60   Wed Nov 18 24:00  Tue Nov 17 18:24  Tue Nov 17 21:31  0 of 20
WO-26104  FTG-2201  120  Thu Nov 19 24:00  Wed Nov 18 15:43  Wed Nov 18 18:00  0 of 20
WO-26105  VB-3107   50   Fri Nov 20 24:00  Fri Nov 20 18:45  Mon Nov 23 12:04  8 of 20
WO-26106  MB-4410   80   Fri Nov 20 24:00  Thu Nov 19 15:03  Thu Nov 19 19:07  0 of 20
WO-26107  VB-3107   30   Mon Nov 23 24:00  Tue Nov 24 11:30  Tue Nov 24 19:26  16 of 20
WO-26108  MB-4410  

What this says:

- Every fitting and every bracket ships on time.
- Valve bodies are late. WO-26110 for Summit Valve is late in every run, by about 5 days.
  WO-26107 and WO-26112 are late in 16 runs of 20. WO-26105 is a coin flip.
- **The bottleneck is the valve-body mill (`mill_VB`).** It is busy 93% of its on-shift
  hours. The next busiest machine is the bracket mill at 31%.

### Test one change: a third machinist, or a second mill

The sweep file names two changes. The twin runs every mix of them on the same 20 sets of
random times, so the comparison is fair.

In [7]:
!twinflow balance shop.yaml --plan data/open-orders.csv --sweep sweeps/machinist-or-mill.json --reps 20

sweep comparison (mean across replications):

  point        on_time_pct  busy_hours  levers
  ------------ ----------- -----------  ------
  point-0000          79.7       776.9  labor.pools[0].headcount=2, locations[mill_VB].capacity=1
  point-0001          95.7       888.5  labor.pools[0].headcount=2, locations[mill_VB].capacity=2
  point-0002          80.3       693.1  labor.pools[0].headcount=3, locations[mill_VB].capacity=1
  point-0003          99.7       569.4  labor.pools[0].headcount=3, locations[mill_VB].capacity=2

sweep <run-id-2>
  dir:     ./runs/<run-id-2>
  summary: ./runs/<run-id-2>/summary.json
  page:    ./runs/<run-id-2>/sweep.html


How to read it: `on_time_pct` is the share of orders on time, averaged over 20 runs.

| Change | On time |
|---|---|
| Today: 2 machinists, 1 valve-body mill | 79.7% |
| Hire a third machinist | 80.3% |
| Add a second valve-body mill | 95.7% |
| Both | 99.7% |

### The state of the shop over time

This run saves the state of the shop every 8 hours, with the money. `states` reads it.
Each state lists the setup of each machine: `idle` means the machine has no job. The saw
and the inspection station (CMM) serve all three parts, so their setup names the part:
`saw_bar`, `saw_block` and `saw_plate`, and `cmm_FTG`, `cmm_VB` and `cmm_MB`.

In [8]:
!twinflow scenario run ridgeline.twin.yaml --out runs/capture --summary

Ran 5 runs (seed 11).
On time:   73% of orders (range 63% to 83%)
Machine hours per run: 886.4
States saved every 8 h. Read them with twinflow states and twinflow markov.
Full result: runs/capture/result.json


In [9]:
!twinflow states runs/capture --view full --top 5

Moves found in runs/capture
Runs found: 5
Episodes: 5
Moves: 290
Time span: 0 to 1.86264e+06
Unique states: 220
Top states (view full), share of moves:
   25.9%  saw_block,cmm_VB,deburr_bench,lathe,mill_bracket,mill_valve
   18.6%  saw_plate,cmm_FTG,deburr_bench,lathe,mill_bracket,mill_valve
   17.2%  saw_plate,cmm_MB,deburr_bench,lathe,mill_bracket,mill_valve
   13.1%  saw_bar,cmm_MB,deburr_bench,lathe,mill_bracket,mill_valve
    7.9%  saw_block,cmm_MB,deburr_bench,lathe,mill_bracket,mill_valve
Ledger by part (all moves):
  revenue        715,900.00
  production     281,200.00
  setup           11,430.00
  holding              0.00
  lost_sales           0.00
  late            29,492.16
  total          393,777.84


In [10]:
!twinflow markov runs/capture --view full --top 4

View: full
Moves: 290
Coverage: 74.8% of moves start in the 4 states below
States:
   25.9%  saw_block,cmm_VB,deburr_bench,lathe,mill_bracket,mill_valve  (sticky)
   18.6%  saw_plate,cmm_FTG,deburr_bench,lathe,mill_bracket,mill_valve  (sticky)
   17.2%  saw_plate,cmm_MB,deburr_bench,lathe,mill_bracket,mill_valve  (sticky)
   13.1%  saw_bar,cmm_MB,deburr_bench,lathe,mill_bracket,mill_valve  (sticky)
Likely moves (9):
   0.92  saw_block,cmm_VB,deburr_bench,lathe,mill_bracket,mill_valve  ->  saw_block,cmm_VB,deburr_bench,lathe,mill_bracket,mill_valve
   0.84  saw_plate,cmm_MB,deburr_bench,lathe,mill_bracket,mill_valve  ->  saw_plate,cmm_MB,deburr_bench,lathe,mill_bracket,mill_valve
   0.78  saw_plate,cmm_FTG,deburr_bench,lathe,mill_bracket,mill_valve  ->  saw_plate,cmm_FTG,deburr_bench,lathe,mill_bracket,mill_valve
   0.76  saw_bar,cmm_MB,deburr_bench,lathe,mill_bracket,mill_valve  ->  saw_bar,cmm_MB,deburr_bench,lathe,mill_bracket,mill_valve
   0.11  saw_bar,cmm_MB,deburr_bench,lathe,mil

In the five most common states every machine has a job set up: the shop is full. The saw
and the CMM change setup as the part mix moves, and each change costs $90 in the ledger
($11,430 over 5 runs). The late fees cost more: about $29,500 over 5 runs, or about $5,900
for each three-week order book.

## (d) Your rules

For a job shop, the written pieces are the change you want to test and the dispatch rule.
The dispatch rule is one line in the shop file (`dispatch: edd` at each mill). The change
to test is a sweep file. The box below saves it, so you can edit it and run it again.

### `sweeps/machinist-or-mill.json`: the changes to test

Each line names one number in the shop file and the values to try.

- `labor.pools[0].headcount` is the number of machinists on each shift (the first pool in
  `shop.yaml`). It tries 2 (today) and 3.
- `locations[mill_VB].capacity` is the number of valve-body mills. It tries 1 (today) and 2.

The twin runs every mix: 2 times 2 is 4 cases. To try a third machinist only, write
`[2, 3]` for the first line and `[1]` for the second.

In [11]:
%%writefile sweeps/machinist-or-mill.json
{"labor.pools[0].headcount": [2, 3], "locations[mill_VB].capacity": [1, 2]}

Overwriting sweeps/machinist-or-mill.json


## (e) Results and the decision

- **Which orders are at risk:** four Summit Valve and Delta Pumps valve-body orders
  (WO-26105, WO-26107, WO-26110, WO-26112). Call those customers today. Every fitting and
  bracket order ships on time.
- **What holds them up:** the valve-body mill. It is busy 93% of its on-shift hours, while
  the bracket mill is about 31% and the lathe about 22%.
- **The decision:** do not hire a third machinist to fix this. It moves on-time from 79.7%
  to 80.3%, because the people are not the constraint. The machine is.
- **Add valve-body mill capacity.** A second VF-2 moves on-time to 95.7%. Late fees now
  run about $5,900 per three-week order book, about $100,000 a year, so a $95,000 mill pays
  back in about a year before counting the extra valve-body sales it allows.
- **This week:** the mill is the only machine that matters. Run it through breaks and
  lunch, and move bracket work off it first.

The run's full report, with each order's finish time and the wait at every machine, is
`report.html` in the run folder printed by the `run` box (for example
`runs/<run id>/report.html`). Open it in a web browser.

## (f) Try it yourself

1. **A Saturday shift.** In `shop.yaml`, add `sat` to the machinists' `days` list (both
   pools, if you want inspection on Saturday too). Run the `run` box again. How many valve
   bodies are still late?
2. **A rush order.** Add a row to `data/open-orders.csv`:
   `WO-26116,Delta Pumps,VB-3107,20,2026-11-18,2026-11-24,1`. Run the `run` box. Which
   orders does the rush push out?
3. **A faster program.** A new tool path cuts the valve body to 38 minutes. In `shop.yaml`,
   change the `mill_VB` rate from `0.0003704` to `0.0004386` (1 / 2280 seconds). Run the
   sweep again. Is the second mill still worth it?